## Load Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/gold/property_listings_geocoded.csv"
)

print(df.shape)
df.head()

(2673, 33)


,listing_id,source,source_listing_id,source_listing_code,url,title,description,listing_type,property_type,project_name,...,longitude,listing_created_at,listing_updated_at,scraped_at,district_original,district_recovery_source,geocode_level,geocode_query,coordinate_source,coordinate_precision
0,realestate.com.kh_246560,realestate.com.kh,246560,NaN,https://www.realestate.com.kh/new-developments...,RARE 1-Bedroom Condo | Sale | Time Square 2 | ...,Property type Condo Title Hard Title Property ...,sale,Condo,Time Square II,...,NaN,2024-10-28T15:59:53.026661+07:00,NaN,2026-08-05T04:26:18+00:00,Toul Kork,NaN,project,"Time Square II, Toul Kork, Phnom Penh, Cambodia",NaN,NaN
1,realestate.com.kh_259255,realestate.com.kh,259255,NaN,https://www.realestate.com.kh/new-developments...,2-Bedroom Condo for Sale at Chipmong Parkland ...,Property type Condo Title Strata Title Propert...,sale,Condo,Chip Mong | Park Land TK Condo,...,NaN,2026-02-18T15:22:57.238626+07:00,NaN,2026-08-05T04:26:18+00:00,Sen Sok,NaN,project,"Chip Mong | Park Land TK Condo, Sen Sok, Phnom...",NaN,NaN
2,realestate.com.kh_248268,realestate.com.kh,248268,NaN,https://www.realestate.com.kh/new-developments...,Rare 3-Bedroom Condominium for sale Time Squar...,Property type Condo Title Strata Title Propert...,sale,Condo,Time Square 306,...,NaN,2025-01-27T17:03:59.484763+07:00,NaN,2026-08-05T04:26:18+00:00,Boeung Keng Kang,NaN,project,"Time Square 306, Boeung Keng Kang, Phnom Penh,...",NaN,NaN
3,realestate.com.kh_255697,realestate.com.kh,255697,NaN,https://www.realestate.com.kh/new-developments...,1-Bedroom Condo for Resale at One Park Residence,Property type Condo Title Hard Title Property ...,sale,Condo,One Park,...,104.905704,2025-09-29T14:49:58.371647+07:00,NaN,2026-08-05T04:26:18+00:00,Daun Penh,NaN,project,"One Park, Daun Penh, Phnom Penh, Cambodia",first_pass,project_building
4,realestate.com.kh_247259,realestate.com.kh,247259,NaN,https://www.realestate.com.kh/new-developments...,Studio Condo for Sale | Tk Star Project,Property type Condo Property ID 247259 Origina...,sale,Condo,TK Star International,...,NaN,2024-12-05T15:43:31.339570+07:00,NaN,2026-08-05T04:26:18+00:00,Toul Kork,NaN,project,"TK Star International, Toul Kork, Phnom Penh, ...",NaN,NaN


## Create the modeling target

Because asking-price distribution is strongly right-skewed, create:

In [2]:
df["log_price_usd"] = np.log1p(
    df["price_usd"]
)

In [3]:
df[
    ["price_usd", "log_price_usd"]
].describe()

,price_usd,log_price_usd
count,2.673000e+03,2673.000000
mean,1.708184e+05,11.623875
std,2.593639e+05,0.806490
min,2.000000e+04,9.903538
25%,6.200000e+04,11.034906
50%,9.500000e+04,11.461643
75%,1.800000e+05,12.100718
max,5.000000e+06,15.424949


## Select the main model features

Based on your EDA, start with:

In [4]:
model_features = [
    "size_m2",
    "bedrooms",
    "bathrooms",
    "unit_floor",
    "district",
    "property_type",
]

In [5]:
target = "log_price_usd"

## Create the modeling dataframe

In [6]:
model_df = df[
    model_features + [target]
].copy()

print(model_df.shape)

model_df.head()

(2673, 7)


,size_m2,bedrooms,bathrooms,unit_floor,district,property_type,log_price_usd
0,35.0,1.0,1.0,24.0,Toul Kork,Condo,11.211834
1,53.0,2.0,2.0,10.0,Sen Sok,Condo,11.482477
2,135.0,3.0,3.0,45.0,Boeung Keng Kang,Condo,12.611541
3,73.0,1.0,1.0,NaN,Daun Penh,Condo,12.072547
4,37.0,0.0,1.0,6.0,Toul Kork,Condo,10.878066


In [7]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2673 entries, 0 to 2672
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   size_m2        2673 non-null   float64
 1   bedrooms       2501 non-null   float64
 2   bathrooms      1960 non-null   float64
 3   unit_floor     2057 non-null   float64
 4   district       2454 non-null   str    
 5   property_type  2673 non-null   str    
 6   log_price_usd  2673 non-null   float64
dtypes: float64(5), str(2)
memory usage: 187.4 KB


## Check missing values again

In [8]:
missing_summary = pd.DataFrame({
    "missing": model_df.isna().sum(),
    "missing_pct": (
        model_df.isna().mean() * 100
    ).round(2)
})

missing_summary

,missing,missing_pct
size_m2,0,0.00
bedrooms,172,6.43
bathrooms,713,26.67
unit_floor,616,23.05
district,219,8.19
property_type,0,0.00
log_price_usd,0,0.00


## save the raw model dataset

Before splitting, save this version:

In [9]:
from pathlib import Path

model_dir = Path(
    "../data/gold/modeling"
)

model_dir.mkdir(
    parents=True,
    exist_ok=True
)

model_df.to_csv(
    model_dir / "model_ready_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

print(model_df.shape)

(2673, 7)


## Train/Test Split

In [10]:
# use scikit-learn to split the data into train and test sets

from sklearn.model_selection import train_test_split

In [11]:
# separate features and target variable

X = model_df.drop(
    columns=["log_price_usd"]
)

y = model_df[
    "log_price_usd"
]

In [12]:
# check shape of X and y

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2673, 6)
y shape: (2673,)


## split out the final test set

First reserve 10% final test data:

In [13]:
# now split the data into train 90% and test 10%

from sklearn.model_selection import train_test_split


X_dev, X_test_final, y_dev, y_test_final = (
    train_test_split(
        X,
        y,
        test_size=0.10,
        random_state=42,
        stratify=X["property_type"],
    )
)

## split out calibration data

We need 10% of the original dataset for calibration.

Since X_dev contains 90%, use:

In [14]:
calibration_fraction = (
    0.10 / 0.90
)

X_dev2, X_calibration, y_dev2, y_calibration = (
    train_test_split(
        X_dev,
        y_dev,
        test_size=calibration_fraction,
        random_state=42,
        stratify=X_dev[
            "property_type"
        ],
    )
)

## split development into train and validation

X_dev2 contains 80% of all data.

In [15]:
validation_fraction = (
    0.10 / 0.80
)

X_train, X_val, y_train, y_val = (
    train_test_split(
        X_dev2,
        y_dev2,
        test_size=validation_fraction,
        random_state=42,
        stratify=X_dev2[
            "property_type"
        ],
    )
)

## Check the sizes

In [16]:
print(
    "Full dataset:",
    len(X)
)

print(
    "Training:",
    len(X_train)
)

print(
    "Validation:",
    len(X_val)
)

print(
    "Calibration:",
    len(X_calibration)
)

print(
    "Final test:",
    len(X_test_final)
)

Full dataset: 2673
Training: 1869
Validation: 268
Calibration: 268
Final test: 268


## verify property_type balance

In [18]:
def check_property_type_distribution(
    name,
    X_split,
):
    print(
        f"\n{name}"
    )

    print(
        X_split[
            "property_type"
        ]
        .value_counts()
    )

    print(
        "\nPercentage:"
    )

    print(
        (
            X_split[
                "property_type"
            ]
            .value_counts(
                normalize=True
            )
            * 100
        ).round(2)
    )


check_property_type_distribution(
    "Full Dataset",
    X,
)

check_property_type_distribution(
    "Training",
    X_train,
)




Full Dataset
property_type
Condo        2482
Penthouse     191
Name: count, dtype: int64

Percentage:
property_type
Condo        92.85
Penthouse     7.15
Name: proportion, dtype: float64

Training
property_type
Condo        1735
Penthouse     134
Name: count, dtype: int64

Percentage:
property_type
Condo        92.83
Penthouse     7.17
Name: proportion, dtype: float64


In [20]:
check_property_type_distribution(
    "Validation",
    X_val,
)

check_property_type_distribution(
    "Calibration",
    X_calibration,
)




Validation
property_type
Condo        249
Penthouse     19
Name: count, dtype: int64

Percentage:
property_type
Condo        92.91
Penthouse     7.09
Name: proportion, dtype: float64

Calibration
property_type
Condo        249
Penthouse     19
Name: count, dtype: int64

Percentage:
property_type
Condo        92.91
Penthouse     7.09
Name: proportion, dtype: float64


In [21]:
check_property_type_distribution(
    "Final Test",
    X_test_final,
)


Final Test
property_type
Condo        249
Penthouse     19
Name: count, dtype: int64

Percentage:
property_type
Condo        92.91
Penthouse     7.09
Name: proportion, dtype: float64


## the price distribution across the four splits

In [23]:
def check_target_distribution(
    name,
    y_split,
):
    price = np.expm1(
        y_split
    )

    print(f"\n{name}")
    print("-" * 50)

    print(
        f"Count  : {len(price)}"
    )

    print(
        f"Mean   : ${np.mean(price):,.0f}"
    )

    print(
        f"Median : ${np.median(price):,.0f}"
    )

    print(
        f"P25    : ${np.percentile(price, 25):,.0f}"
    )

    print(
        f"P75    : ${np.percentile(price, 75):,.0f}"
    )

    print(
        f"P95    : ${np.percentile(price, 95):,.0f}"
    )

    print(
        f"Max    : ${np.max(price):,.0f}"
    )


check_target_distribution(
    "Full Dataset",
    y,
)

check_target_distribution(
    "Training",
    y_train,
)




Full Dataset
--------------------------------------------------
Count  : 2673
Mean   : $170,818
Median : $95,000
P25    : $62,000
P75    : $180,000
P95    : $520,000
Max    : $5,000,000

Training
--------------------------------------------------
Count  : 1869
Mean   : $169,796
Median : $93,100
P25    : $60,000
P75    : $180,000
P95    : $518,000
Max    : $3,121,250


In [ ]:
check_target_distribution(
    "Validation",
    y_val,
)

check_target_distribution(
    "Calibration",
    y_calibration,
)




Validation
--------------------------------------------------
Count  : 268
Mean   : $169,818
Median : $95,000
P25    : $60,000
P75    : $168,129
P95    : $581,282
Max    : $2,470,000

Calibration
--------------------------------------------------
Count  : 268
Mean   : $183,370
Median : $98,515
P25    : $69,000
P75    : $159,999
P95    : $509,750
Max    : $5,000,000

Final Test
--------------------------------------------------
Count  : 268
Mean   : $166,396
Median : $98,000
P25    : $68,000
P75    : $200,000
P95    : $520,530
Max    : $1,300,000


In [25]:
check_target_distribution(
    "Final Test",
    y_test_final,
)


Final Test
--------------------------------------------------
Count  : 268
Mean   : $166,396
Median : $98,000
P25    : $68,000
P75    : $200,000
P95    : $520,530
Max    : $1,300,000


In [27]:
X_test_final
y_test_final

1869    11.903284
2140    12.013707
2262    13.518472
2348    11.367911
1354    11.775297
          ...    
1132    13.458837
2009    10.545368
1335    12.706851
2391    12.180760
1351    11.002117
Name: log_price_usd, Length: 268, dtype: float64

## One final distribution check

In [26]:
price_bins = [
    0,
    100_000,
    200_000,
    350_000,
    500_000,
    1_000_000,
    np.inf,
]

price_labels = [
    "<$100K",
    "$100K-$200K",
    "$200K-$350K",
    "$350K-$500K",
    "$500K-$1M",
    ">$1M",
]


def price_band_distribution(
    name,
    y_split,
):
    prices = np.expm1(
        y_split
    )

    bands = pd.cut(
        prices,
        bins=price_bins,
        labels=price_labels,
        include_lowest=True,
    )

    result = (
        bands
        .value_counts(
            normalize=True
        )
        .sort_index()
        * 100
    )

    return result.rename(
        name
    )


price_distribution_check = pd.concat(
    [
        price_band_distribution(
            "Full",
            y,
        ),

        price_band_distribution(
            "Train",
            y_train,
        ),

        price_band_distribution(
            "Validation",
            y_val,
        ),

        price_band_distribution(
            "Calibration",
            y_calibration,
        ),

        price_band_distribution(
            "Final Test",
            y_test_final,
        ),
    ],
    axis=1,
)

price_distribution_check.round(2)

,Full,Train,Validation,Calibration,Final Test
log_price_usd,,,,,
<$100K,53.27,54.09,51.12,51.49,51.49
$100K-$200K,25.63,24.61,29.48,30.60,23.88
$200K-$350K,12.64,12.89,11.19,9.33,15.67
$350K-$500K,3.18,3.21,2.24,3.36,3.73
$500K-$1M,3.93,3.80,4.48,3.36,4.85
>$1M,1.35,1.39,1.49,1.87,0.37


## Retrain Linear Regression

Use existing preprocessing approach. Numerical features:

In [28]:
numeric_features = [
    "size_m2",
    "bedrooms",
    "bathrooms",
    "unit_floor",
]

categorical_features = [
    "district",
    "property_type",
]

In [35]:
# create the preprocessing pipelines for numeric and categorical features

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.linear_model import LinearRegression


# =========================================================
# NUMERIC PREPROCESSING
# =========================================================

linear_numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
        ),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
])


# =========================================================
# CATEGORICAL PREPROCESSING
# =========================================================

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown",
        ),
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
        ),
    ),
])


# =========================================================
# PREPROCESSOR
# =========================================================

linear_preprocessor = ColumnTransformer([
    (
        "num",
        linear_numeric_pipeline,
        numeric_features,
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features,
    ),
])

In [36]:
# build the model 

linear_model_new = Pipeline([
    (
        "preprocessor",
        linear_preprocessor,
    ),
    (
        "model",
        LinearRegression(),
    ),
])

In [37]:
# using training set only:

X_train
y_train

926     10.687412
1974    11.082158
422     11.532738
2248    11.034906
715     11.461643
          ...    
756     10.878066
1798    11.744045
1393    11.300054
1032    10.596660
1905    12.824714
Name: log_price_usd, Length: 1869, dtype: float64

In [38]:
linear_model_new.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['size_m2','bedrooms','bathrooms','unit_floor','district','property_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of column

## Predict Validation

Do not touch X_test_final.

In [39]:
linear_val_pred_log = (
    linear_model_new.predict(
        X_val
    )
)

In [41]:
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)


def evaluate_model(
    model_name,
    y_true_log,
    y_pred_log,
):
    # Convert log prices back to USD
    y_true_usd = np.expm1(
        np.asarray(y_true_log)
    )

    y_pred_usd = np.expm1(
        np.asarray(y_pred_log)
    )

    # -----------------------------
    # USD metrics
    # -----------------------------

    rmse = np.sqrt(
        mean_squared_error(
            y_true_usd,
            y_pred_usd,
        )
    )

    mae = mean_absolute_error(
        y_true_usd,
        y_pred_usd,
    )

    mape = (
        mean_absolute_percentage_error(
            y_true_usd,
            y_pred_usd,
        )
        * 100
    )

    r2 = r2_score(
        y_true_usd,
        y_pred_usd,
    )

    # -----------------------------
    # Log-space metrics
    # -----------------------------

    log_rmse = np.sqrt(
        mean_squared_error(
            y_true_log,
            y_pred_log,
        )
    )

    log_mae = mean_absolute_error(
        y_true_log,
        y_pred_log,
    )

    log_r2 = r2_score(
        y_true_log,
        y_pred_log,
    )

    return {
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2,
        "Log_RMSE": log_rmse,
        "Log_MAE": log_mae,
        "Log_R2": log_r2,
    }

In [42]:
linear_validation_result = evaluate_model(
    "Linear Regression - Validation",
    y_val,
    linear_val_pred_log,
)

pd.DataFrame([
    linear_validation_result
])

,Model,RMSE,MAE,MAPE,R2,Log_RMSE,Log_MAE,Log_R2
0,Linear Regression - Validation,134816.178031,53659.831854,32.949261,0.677208,0.399603,0.308614,0.757617


In [44]:
# the training results

linear_train_pred_log = (
    linear_model_new.predict(
        X_train
    )
)

linear_train_result = evaluate_model(
    "Linear Regression - Train",
    y_train,
    linear_train_pred_log,
)

In [45]:
# compare the training and validation results

linear_train_val_comparison = pd.DataFrame([
    linear_train_result,
    linear_validation_result,
])

linear_train_val_comparison

,Model,RMSE,MAE,MAPE,R2,Log_RMSE,Log_MAE,Log_R2
0,Linear Regression - Train,256259.598665,66531.907318,33.426491,-0.054155,0.398220,0.310900,0.758839
1,Linear Regression - Validation,134816.178031,53659.831854,32.949261,0.677208,0.399603,0.308614,0.757617


## Random Forest

In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


numeric_features = [
    "size_m2",
    "bedrooms",
    "bathrooms",
    "unit_floor",
]

categorical_features = [
    "district",
    "property_type",
]


tree_numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
        ),
    ),
])


tree_categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown",
        ),
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
        ),
    ),
])


tree_preprocessor = ColumnTransformer([
    (
        "num",
        tree_numeric_pipeline,
        numeric_features,
    ),
    (
        "cat",
        tree_categorical_pipeline,
        categorical_features,
    ),
])

## Create stratified CV folds

In [47]:
from sklearn.model_selection import StratifiedKFold


stratified_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


cv_splits = list(
    stratified_cv.split(
        X_train,
        X_train["property_type"],
    )
)

This means each CV fold should contain approximately the same:

Condo       ~92.8%
Penthouse    ~7.2%

as training data.

## Build Random Forest

In [48]:
from sklearn.ensemble import RandomForestRegressor


rf_pipeline_new = Pipeline([
    (
        "preprocessor",
        tree_preprocessor,
    ),
    (
        "model",
        RandomForestRegressor(
            random_state=42,
            n_jobs=1,
        ),
    ),
])

In [49]:
# use tuning:

rf_param_distributions = {
    "model__n_estimators": [
        300,
        500,
        700,
        900,
    ],

    "model__max_depth": [
        None,
        10,
        15,
        20,
        25,
    ],

    "model__min_samples_split": [
        2,
        5,
        10,
        15,
    ],

    "model__min_samples_leaf": [
        1,
        2,
        4,
        6,
    ],

    "model__max_features": [
        0.5,
        0.7,
        0.9,
        1.0,
    ],

    "model__bootstrap": [
        True,
    ],
}

In [50]:
from sklearn.model_selection import RandomizedSearchCV


rf_search_new = RandomizedSearchCV(
    estimator=rf_pipeline_new,
    param_distributions=rf_param_distributions,
    n_iter=30,
    scoring="neg_mean_squared_error",
    cv=cv_splits,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)


rf_search_new.fit(
    X_train,
    y_train,
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__bootstrap': [True], 'model__max_depth': [None, 10, ...], 'model__max_features': [0.5, 0.7, ...], 'model__min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...shape=(1495,)), ...), (array([ 0, ...shape=(1495,)), ...), ...]"
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores o

In [51]:
print("Best Parameters:")
print(
    rf_search_new.best_params_
)


best_rf_cv_mse = (
    -rf_search_new.best_score_
)

best_rf_cv_log_rmse = np.sqrt(
    best_rf_cv_mse
)

print(
    "\nBest CV Log RMSE:",
    best_rf_cv_log_rmse,
)

Best Parameters:
{'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': 0.5, 'model__max_depth': 25, 'model__bootstrap': True}

Best CV Log RMSE: 0.34163123260943334


In [52]:
# get the best model 

best_rf_new = (
    rf_search_new.best_estimator_
)

## Evaluate Train

In [53]:
rf_train_pred_log = (
    best_rf_new.predict(
        X_train
    )
)

rf_train_result = evaluate_model(
    "Random Forest - Train",
    y_train,
    rf_train_pred_log,
)

## Evaluate Validation

In [54]:
rf_val_pred_log = (
    best_rf_new.predict(
        X_val
    )
)

rf_validation_result = evaluate_model(
    "Random Forest - Validation",
    y_val,
    rf_val_pred_log,
)

In [56]:
# compare

rf_train_val_comparison = pd.DataFrame([
    rf_train_result,
    rf_validation_result,
])

rf_train_val_comparison

,Model,RMSE,MAE,MAPE,R2,Log_RMSE,Log_MAE,Log_R2
0,Random Forest - Train,80689.349093,30013.414480,16.500960,0.895485,0.223415,0.161646,0.924092
1,Random Forest - Validation,106100.185383,44822.060589,26.973266,0.800073,0.348773,0.257765,0.815358


## train XGBoost

In [57]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV


xgb_pipeline_new = Pipeline([
    (
        "preprocessor",
        tree_preprocessor,
    ),
    (
        "model",
        XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            n_jobs=1,
        ),
    ),
])

In [58]:
xgb_param_distributions = {
    "model__n_estimators": [
        200,
        300,
        400,
        500,
        700,
    ],

    "model__learning_rate": [
        0.02,
        0.03,
        0.05,
        0.07,
        0.10,
    ],

    "model__max_depth": [
        2,
        3,
        4,
        5,
        6,
    ],

    "model__min_child_weight": [
        1,
        3,
        5,
        8,
        10,
    ],

    "model__subsample": [
        0.6,
        0.7,
        0.8,
        0.9,
        1.0,
    ],

    "model__colsample_bytree": [
        0.6,
        0.7,
        0.8,
        0.9,
        1.0,
    ],

    "model__reg_alpha": [
        0,
        0.01,
        0.1,
        0.5,
        1,
    ],

    "model__reg_lambda": [
        1,
        2,
        5,
        10,
    ],
}

In [59]:
xgb_search_new = RandomizedSearchCV(
    estimator=xgb_pipeline_new,
    param_distributions=xgb_param_distributions,
    n_iter=30,
    scoring="neg_mean_squared_error",
    cv=cv_splits,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

xgb_search_new.fit(
    X_train,
    y_train,
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__colsample_bytree': [0.6, 0.7, ...], 'model__learning_rate': [0.02, 0.03, ...], 'model__max_depth': [2, 3, ...], 'model__min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...shape=(1495,)), ...), (array([ 0, ...shape=(1495,)), ...), ...]"
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However compu

In [60]:
print("Best Parameters:")
print(
    xgb_search_new.best_params_
)

best_xgb_cv_mse = (
    -xgb_search_new.best_score_
)

best_xgb_cv_log_rmse = np.sqrt(
    best_xgb_cv_mse
)

print(
    "\nBest CV Log RMSE:",
    best_xgb_cv_log_rmse,
)

Best Parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 5, 'model__reg_alpha': 1, 'model__n_estimators': 500, 'model__min_child_weight': 1, 'model__max_depth': 5, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.9}

Best CV Log RMSE: 0.34353564790585106


In [61]:
# evaluate Train and Validation:

best_xgb_new = (
    xgb_search_new.best_estimator_
)


xgb_train_pred_log = (
    best_xgb_new.predict(
        X_train
    )
)

xgb_val_pred_log = (
    best_xgb_new.predict(
        X_val
    )
)


xgb_train_result = evaluate_model(
    "XGBoost - Train",
    y_train,
    xgb_train_pred_log,
)

xgb_validation_result = evaluate_model(
    "XGBoost - Validation",
    y_val,
    xgb_val_pred_log,
)


xgb_train_val_comparison = pd.DataFrame([
    xgb_train_result,
    xgb_validation_result,
])

xgb_train_val_comparison

,Model,RMSE,MAE,MAPE,R2,Log_RMSE,Log_MAE,Log_R2
0,XGBoost - Train,73316.626829,32894.870287,20.303287,0.913712,0.263441,0.197030,0.894457
1,XGBoost - Validation,107444.120449,43293.554279,27.672579,0.794976,0.347902,0.258882,0.816279


## refit XGBoost on Train + Validation

In [62]:
import pandas as pd


X_train_final = pd.concat(
    [
        X_train,
        X_val,
    ],
    axis=0,
)

y_train_final = pd.concat(
    [
        y_train,
        y_val,
    ],
    axis=0,
)


print(
    "Final training X:",
    X_train_final.shape
)

print(
    "Final training y:",
    y_train_final.shape
)

Final training X: (2137, 6)
Final training y: (2137,)


In [63]:
# selected tuned XGBoost model

from sklearn.base import clone


final_xgb_model = clone(
    best_xgb_new
)

In [64]:
final_xgb_model.fit(
    X_train_final,
    y_train_final,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['size_m2','bedrooms','bathrooms','unit_floor','district','property_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of column

## After fitting, use Calibration

Do not evaluate X_test_final yet.

In [65]:
calibration_pred_log = (
    final_xgb_model.predict(
        X_calibration
    )
)

In [66]:
# Calculate calibration residuals:

calibration_residuals = np.abs(
    y_calibration.to_numpy()
    - calibration_pred_log
)

In [67]:
print(
    "Calibration rows:",
    len(calibration_residuals)
)

print(
    "Median calibration error:",
    np.median(calibration_residuals)
)

print(
    "Mean calibration error:",
    np.mean(calibration_residuals)
)

Calibration rows: 268
Median calibration error: 0.19356016187736902
Mean calibration error: 0.2580459664903401


In [68]:
# calculate the new 80% conformal q_hat:

confidence_level = 0.80

n_cal = len(
    calibration_residuals
)

quantile_level = (
    np.ceil(
        (n_cal + 1)
        * confidence_level
    )
    / n_cal
)

quantile_level = min(
    quantile_level,
    1.0,
)

q_hat_80_final = np.quantile(
    calibration_residuals,
    quantile_level,
    method="higher",
)


print(
    f"Final 80% q_hat: "
    f"{q_hat_80_final:.4f}"
)

Final 80% q_hat: 0.4153


## Unlock the Final Test once

In [69]:
# =========================================================
# FINAL TEST — POINT PREDICTION
# =========================================================

final_test_pred_log = (
    final_xgb_model.predict(
        X_test_final
    )
)


final_test_result = evaluate_model(
    "Final XGBoost",
    y_test_final,
    final_test_pred_log,
)


pd.DataFrame([
    final_test_result
])

,Model,RMSE,MAE,MAPE,R2,Log_RMSE,Log_MAE,Log_R2
0,Final XGBoost,122245.874742,53075.18253,26.079751,0.552943,0.337425,0.256204,0.816424


## Evaluate the final 80% price range

In [70]:
# =========================================================
# FINAL 80% CONFORMAL INTERVAL
# =========================================================

final_lower_log = (
    final_test_pred_log
    - q_hat_80_final
)

final_upper_log = (
    final_test_pred_log
    + q_hat_80_final
)


final_pred_usd = np.expm1(
    final_test_pred_log
)

final_lower_usd = np.expm1(
    final_lower_log
)

final_upper_usd = np.expm1(
    final_upper_log
)

final_actual_usd = np.expm1(
    y_test_final.to_numpy()
)

In [71]:
final_covered = (
    (final_actual_usd >= final_lower_usd)
    &
    (final_actual_usd <= final_upper_usd)
)


final_coverage = (
    final_covered.mean()
)


print(
    "Final 80% interval coverage:",
    f"{final_coverage * 100:.2f}%"
)

Final 80% interval coverage: 77.61%


In [72]:
final_interval_width = (
    final_upper_usd
    - final_lower_usd
)


print(
    "Median interval width:",
    f"${np.median(final_interval_width):,.0f}"
)

print(
    "Mean interval width:",
    f"${np.mean(final_interval_width):,.0f}"
)

Median interval width: $82,768
Mean interval width: $139,065


In [73]:
print(
    "Median lower ratio:",
    f"{np.median(final_lower_usd / final_pred_usd):.2f}"
)

print(
    "Median upper ratio:",
    f"{np.median(final_upper_usd / final_pred_usd):.2f}"
)

Median lower ratio: 0.66
Median upper ratio: 1.51


## Put the final results together

In [74]:
final_evaluation = pd.DataFrame({
    "actual_price": final_actual_usd,
    "predicted_price": final_pred_usd,
    "lower_bound": final_lower_usd,
    "upper_bound": final_upper_usd,
    "covered": final_covered,
    "interval_width": final_interval_width,
})

final_evaluation.head(10)

,actual_price,predicted_price,lower_bound,upper_bound,covered,interval_width
0,147750.0,126191.828125,83300.877123,191166.707942,True,107865.830819
1,165000.0,232050.000000,153179.536298,351529.758211,True,198350.221913
2,743014.0,334788.500000,220998.828407,507166.810756,False,286167.982349
3,86500.0,85403.171875,56375.642806,129376.559773,True,73000.916967
4,130000.0,60784.246094,40124.301754,92081.702477,False,51957.400722
5,82000.0,44162.121094,29151.773313,66901.080083,False,37749.306770
6,30000.0,35775.632812,23615.719550,54196.507925,True,30580.788375
7,38000.0,41784.855469,27582.502646,63299.794172,True,35717.291526
8,85000.0,72349.500000,47758.708433,109601.740160,True,61843.031727
9,410000.0,267955.250000,176881.158660,405922.107797,False,229040.949136


## Now save the real final production model

In [75]:
from pathlib import Path
import json
import joblib


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


MODELS_DIR = (
    PROJECT_ROOT / "models"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================
# SAVE FINAL MODEL
# =========================================================

FINAL_MODEL_PATH = (
    MODELS_DIR
    / "propertylens_xgboost_final.joblib"
)


joblib.dump(
    final_xgb_model,
    FINAL_MODEL_PATH,
)


# =========================================================
# SAVE FINAL METADATA
# =========================================================

FINAL_METADATA_PATH = (
    MODELS_DIR
    / "propertylens_xgboost_final_metadata.json"
)


final_metadata = {
    "model_name":
        "PP PropertyLens Final XGBoost",

    "target":
        "log_price_usd",

    "features": [
        "size_m2",
        "bedrooms",
        "bathrooms",
        "unit_floor",
        "district",
        "property_type",
    ],

    "training_rows":
        int(len(X_train_final)),

    "calibration_rows":
        int(len(X_calibration)),

    "final_test_rows":
        int(len(X_test_final)),

    "interval_level":
        0.80,

    "q_hat":
        float(q_hat_80_final),

    "final_test_metrics": {
        "rmse_usd":
            float(
                final_test_result["RMSE"]
            ),

        "mae_usd":
            float(
                final_test_result["MAE"]
            ),

        "mape_percent":
            float(
                final_test_result["MAPE"]
            ),

        "r2_usd":
            float(
                final_test_result["R2"]
            ),

        "log_rmse":
            float(
                final_test_result["Log_RMSE"]
            ),

        "log_mae":
            float(
                final_test_result["Log_MAE"]
            ),

        "log_r2":
            float(
                final_test_result["Log_R2"]
            ),
    },

    "interval_validation": {
        "observed_coverage":
            float(final_coverage),

        "median_width_usd":
            float(
                np.median(
                    final_interval_width
                )
            ),

        "mean_width_usd":
            float(
                np.mean(
                    final_interval_width
                )
            ),
    },
}


with open(
    FINAL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_metadata,
        file,
        indent=4,
    )


print(
    "Model saved to:",
    FINAL_MODEL_PATH
)

print(
    "Metadata saved to:",
    FINAL_METADATA_PATH
)

Model saved to: d:\Internship\Data Insign Cambodia\pp-propertylens\models\propertylens_xgboost_final.joblib
Metadata saved to: d:\Internship\Data Insign Cambodia\pp-propertylens\models\propertylens_xgboost_final_metadata.json
